# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmadrifayee-cmyk/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Classification, feeding a ranking/scoring output.**

At its core, Lane 2 is a **binary classification** problem: for each page, predict
whether it belongs to the "needs review" class (currently proxied by
`trend_direction == "down"`). The model outputs a probability, and that
probability is what turns this into a **ranked review queue** — pages are sorted
by predicted probability, and the top-K become the action list. So the task type
is classification at the core, with scoring/ranking as the downstream use of the
model's output, not a separate task.


In [1]:
import pandas as pd
import numpy as np
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ahmadrifayee-cmyk/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Task type check: is this binary classification?")
print("Unique values in trend_direction:", df["trend_direction"].unique())
print("This confirms a binary target is derivable (down vs. not-down),")
print("which is what makes this a classification task at its core.")

Task type check: is this binary classification?
Unique values in trend_direction: ['down' 'stable' 'new' 'up' 'flat']
This confirms a binary target is derivable (down vs. not-down),
which is what makes this a classification task at its core.


## 2. Target or proxy

**Target used here: `is_declining_label = (trend_direction == "down")`.**

This label comes from the **current window**, not a future observed outcome —
it's a proxy, not a true forward-looking target. `trend_direction` is itself a
derived bucket calculated from recent performance, not something that happened
*after* a decision point. A stronger version of this target (for the full
capstone) would instead be: features from a prior 90-day window predicting
decline over the *next* 30 days — a genuine future outcome. I'm using the
current-window proxy here because it's what the starter dataset provides, but
I'm naming it honestly as a proxy, not treating it as ground truth.

In [2]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Target distribution (current-window proxy label):")
print(df["is_declining_label"].value_counts())
print(f"\nPositive rate: {df['is_declining_label'].mean():.1%}")
print("\nThis is a PROXY label (current trend bucket), not a future outcome.")
print("A stronger capstone target would be: prior-90d features -> next-30d decline.")

Target distribution (current-window proxy label):
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Positive rate: 54.2%

This is a PROXY label (current trend bucket), not a future outcome.
A stronger capstone target would be: prior-90d features -> next-30d decline.


## 3. Success metric

**Metric: Precision@50.**

Since the output is a ranked review queue and the content team can only act on a
limited number of pages per cycle, the metric has to match how the list is
actually used — not generic accuracy. Precision@50 asks: of the top 50 pages the
model ranks highest, how many are actually positive (declining) by the label
definition? This directly reflects reviewer capacity and wasted-time risk. I'm
choosing 50 as a stand-in for "a realistic weekly review capacity," which I can
adjust later once I know the actual team's bandwidth.

In [6]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

# Baseline sanity check: a naive score using just impressions_90d
naive_score = df["impressions_90d"].fillna(0)
baseline_p50 = precision_at_k(naive_score, df["is_declining_label"], 50)

print(f"Precision@50 using a naive 'sort by impressions' baseline: {baseline_p50:.3f}")
print(f"That's about {baseline_p50*50:.0f} of the top 50 pages actually declining.")
print("This gives a floor to beat once a real model is trained (Notebook 02 showed")
print("the starter pipeline's random forest reaches 0.740 Precision@50 on this data).")

print("\nNote: this naive impressions-only baseline (above) is different from")
print("the starter pipeline's hand-written rule (0.240) used in Section 5 --")
print("that rule combines staleness + visibility conditions, not just impressions alone.")

Precision@50 using a naive 'sort by impressions' baseline: 0.420
That's about 21 of the top 50 pages actually declining.
This gives a floor to beat once a real model is trained (Notebook 02 showed
the starter pipeline's random forest reaches 0.740 Precision@50 on this data).

Note: this naive impressions-only baseline (above) is different from
the starter pipeline's hand-written rule (0.240) used in Section 5 --
that rule combines staleness + visibility conditions, not just impressions alone.


## 4. The unit of analysis, as a real dataframe

**One row = one content page** (`content_id`), for one client, describing its
last-known performance snapshot (impressions, clicks, position, age, freshness,
engagement) over a 90-day window.

In [4]:
print("Shape:", df.shape)
print("\nOne row = one content page. Example rows:")
display_cols = ["content_id", "impressions_90d", "sessions_90d",
                 "avg_position", "content_age_days", "trend_direction",
                 "is_declining_label"]
print(df[display_cols].head(5).to_string(index=False))

print("\nSketch of the target column alone:")
print(df["is_declining_label"].head(10).to_list())

Shape: (30000, 45)

One row = one content page. Example rows:
          content_id  impressions_90d  sessions_90d  avg_position  content_age_days trend_direction  is_declining_label
content_304f48230142             3803            17          10.6               187            down                   1
content_a1fb4e703a9e            15320             9          20.3               445            down                   1
content_9aa793d4d895            12581            11          36.5               141            down                   1
content_331d6c4de07b            11751            78           6.2               463          stable                   0
content_d99b7a2d90ca            19140           145          44.0               263            down                   1

Sketch of the target column alone:
[1, 1, 1, 0, 1, 1, 1, 0, 1, 1]


## 5. Why ML beats a fixed rule here

A fixed rule (e.g., "flag if `days_since_last_update >= 180` AND
`impressions_90d >= 500`") can only combine a couple of thresholds a human
picked by hand. Real pages decline for different combinations of reasons —
staleness alone, or position drift alone, or engagement drop alone, or some mix —
and a fixed rule can't weigh these interactions or adjust how much each signal
matters. This was already demonstrated in Notebook 02: the hand-written rule
reached Precision@50 of 0.240, while a random forest model reached 0.740 on the
same data — roughly **3x more true positives in the top 50** than the rule alone
achieves. That gap is the evidence that this pattern is too multi-dimensional
for a simple if-statement, and that a model that can learn interactions between
signals adds real value here.

In [5]:
print("Evidence from the starter pipeline (outputs/model_report.md):")
print(f"{'Method':<20}{'Precision@50':>15}")
print(f"{'Hand-written rule':<20}{0.240:>15.3f}")
print(f"{'Random forest':<20}{0.740:>15.3f}")
print(f"\nLift: {0.740/0.240:.1f}x more correct pages in the top 50 with a learned model.")

Evidence from the starter pipeline (outputs/model_report.md):
Method                 Precision@50
Hand-written rule             0.240
Random forest                 0.740

Lift: 3.1x more correct pages in the top 50 with a learned model.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.